# 03 — Region proposal por HSV

Probamos el módulo `src/region_proposal` sobre imágenes reales. Es la parte más importante de la fase clásica: aquí es donde decidimos **dónde mirar** en la imagen para luego clasificar (en F3).

**Objetivo**: ver qué cajas candidatas genera el sistema y afinar empíricamente los parámetros.

**Pre-requisitos**: dataset Freiburg descargado y notebook 02 entendido.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from src.preprocessing import preprocess
from src.region_proposal import (
    propose_regions, draw_proposals,
    build_mask, clean_mask, boxes_from_mask,
    HSV_RANGES, LOW_SATURATION_RANGE,
)
from src.utils.io_utils import load_image, list_images

DATA_ROOT = Path('../data/external/klasson_flat')
plt.rcParams['figure.dpi'] = 90

## 1. Cargar una imagen de prueba

Empezamos con algo sencillo: una imagen con un producto bien visible y de color saturado.

In [ ]:
# Cambia esta categoría para probar con distintos tipos de producto
category = 'SODA'
images = list_images(DATA_ROOT / category)

img = load_image(images[0])
img_pre = preprocess(img)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(img_pre); axes[1].set_title('Preprocesada'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## 2. Pipeline completo

Llamamos a `propose_regions` y dibujamos el resultado. Esta es la salida que usaremos en el resto del proyecto.

In [ ]:
proposals = propose_regions(img_pre)
print(f'Generadas {len(proposals)} propuestas')

# Top 5
for p in proposals[:5]:
    print(f'  {p.category:10s}  box=({p.x},{p.y},{p.w},{p.h})  area={p.area}  ar={p.aspect_ratio:.2f}')

annotated = draw_proposals(img_pre, proposals)
plt.figure(figsize=(8, 6))
plt.imshow(annotated)
plt.title(f'{len(proposals)} propuestas')
plt.axis('off')
plt.tight_layout(); plt.show()

## 3. Desglose por etapas

Ahora vemos paso a paso lo que hace `propose_regions` por dentro. Útil para diagnosticar problemas.

In [ ]:
# Convertir a HSV
bgr = cv2.cvtColor(img_pre, cv2.COLOR_RGB2BGR)
hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)

# Visualizamos cada máscara de categoría: cruda y limpia
categories_to_show = list(HSV_RANGES.keys())
n = len(categories_to_show)
fig, axes = plt.subplots(n, 3, figsize=(11, 2.4 * n))

for i, cat in enumerate(categories_to_show):
    ranges = HSV_RANGES[cat]
    raw_mask = build_mask(hsv, ranges)
    clean = clean_mask(raw_mask)
    overlay = img_pre.copy()
    overlay[clean == 0] = (overlay[clean == 0] * 0.3).astype(np.uint8)

    axes[i, 0].imshow(raw_mask, cmap='gray'); axes[i, 0].set_title(f'{cat} — máscara cruda', fontsize=9)
    axes[i, 1].imshow(clean, cmap='gray');   axes[i, 1].set_title(f'{cat} — limpia', fontsize=9)
    axes[i, 2].imshow(overlay);              axes[i, 2].set_title(f'{cat} — sobre la imagen', fontsize=9)
    for ax in axes[i]:
        ax.axis('off')

plt.tight_layout()
plt.show()

## 4. Galería sobre varias categorías

Para hacerse una idea de cómo de bien (o mal) funciona el region proposal en general.

In [ ]:
categories_test = ['SODA', 'JUICE', 'TOMATO_SAUCE', 'CEREAL', 'CHIPS', 'PASTA']

fig, axes = plt.subplots(len(categories_test), 2, figsize=(8, 3.5 * len(categories_test)))
for i, cat in enumerate(categories_test):
    cat_dir = DATA_ROOT / cat
    if not cat_dir.is_dir():
        continue
    imgs = list_images(cat_dir)
    if not imgs:
        continue
    img = load_image(imgs[0])
    img_pre = preprocess(img)
    proposals = propose_regions(img_pre)
    annotated = draw_proposals(img_pre, proposals)

    axes[i, 0].imshow(img_pre); axes[i, 0].set_title(f'{cat} — preprocesada', fontsize=10); axes[i, 0].axis('off')
    axes[i, 1].imshow(annotated); axes[i, 1].set_title(f'{cat} — {len(proposals)} cajas', fontsize=10); axes[i, 1].axis('off')

plt.tight_layout()
plt.show()

## 5. Sandbox: ajusta parámetros

Aquí puedes ir cambiando los parámetros y reejecutar para ver el efecto. Cuando encuentres una combinación que te guste, la dejamos como default en el código.

In [ ]:
img = load_image(list_images(DATA_ROOT / 'SODA')[0])
img_pre = preprocess(img)

# Prueba a cambiar estos valores
params = dict(
    kernel_size=5,
    closing_iters=2,
    opening_iters=1,
    min_area_ratio=0.001,
    max_area_ratio=0.5,
    min_aspect=0.2,
    max_aspect=5.0,
    iou_threshold=0.4,
    use_low_saturation=True,
)

proposals = propose_regions(img_pre, **params)
annotated = draw_proposals(img_pre, proposals)

plt.figure(figsize=(9, 6))
plt.imshow(annotated)
plt.title(f'{len(proposals)} propuestas con parámetros actuales')
plt.axis('off')
plt.tight_layout()
plt.show()

print('Resumen por categoría:')
from collections import Counter
for cat, n in Counter(p.category for p in proposals).items():
    print(f'  {cat:10s} {n}')

## 6. Cosas a observar y problemas típicos

Cuando ejecutes esto vas a ver tanto aciertos como fallos. Es lo esperado en F2: el region proposal **no tiene que ser perfecto**, solo tiene que producir un conjunto de cajas que contenga a los productos reales (alto recall). En F3 el clasificador descartará las cajas falsas.

Problemas típicos que verás:

- **Cajas duplicadas**: ajusta `iou_threshold` (más bajo = más NMS).
- **Cajas microscópicas**: sube `min_area_ratio`.
- **Una caja que cubre toda la imagen**: baja `max_area_ratio`.
- **Categorías mal separadas**: revisa `hsv_ranges.py`, los rangos son orientativos.
- **Productos no detectados**: prueba a desactivar `use_low_saturation` o ajustar los rangos.

**Próximo paso (F3)**: extracción de features (HOG + histograma HSV) sobre cada caja y entrenamiento del clasificador SVM.